# Mask2Former: Two-Stage Training (One Notebook)

Two-stage train/val pipeline using HuggingFace Mask2Former.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import yaml

PROJECT_ROOT = Path('.').resolve()
VENV_PYTHON = PROJECT_ROOT / '.venv' / 'Scripts' / 'python.exe'
if not VENV_PYTHON.exists():
    raise FileNotFoundError(f'Python from .venv not found: {VENV_PYTHON}')

print('Project root:', PROJECT_ROOT)
print('Python:', VENV_PYTHON)


In [ ]:
# Paths and logs
SPLITS_DIR = Path('data/splits')
LOG_STAGE1 = Path('runs/mask2former_stage1_train.log')
LOG_STAGE2 = Path('runs/mask2former_stage2_train.log')
RUN_DIR = Path('runs/mask2former_two_stage')
RUN_DIR.mkdir(parents=True, exist_ok=True)
STAGE1_BEST = RUN_DIR / 'stage1_best.pt'
STAGE2_BEST = RUN_DIR / 'stage2_best.pt'


In [ ]:
# Helper to run shell commands with notebook-friendly progress + realtime curves
import re
from tqdm.auto import tqdm
from IPython.display import display
import matplotlib.pyplot as plt


def run_cmd(
    cmd,
    cwd='.',
    log_path=None,
    epoch_total=None,
    quiet_tqdm_lines=True,
    live_plots=True,
    live_plot_every=1,
):
    print('\n>>>', ' '.join(cmd))

    log_f = None
    if log_path is not None:
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log_f = log_path.open('w', encoding='utf-8')
        print('logging to:', log_path)

    env = os.environ.copy()
    env['PYTHONIOENCODING'] = 'utf-8'

    train_pbar = None
    val_pbar = None
    current_epoch = None

    hist_train_loss = []
    hist_val_f1 = []
    plot_handle = None

    train_re = re.compile(r"Epoch\s+(\d+)/(\d+)\s+\[train\]:.*?\|\s*(\d+)/(\d+)\s*\[.*loss=([0-9.]+)")
    val_re = re.compile(r"Epoch\s+(\d+)/(\d+)\s+\[val\]:.*?\|\s*(\d+)/(\d+)\s*\[")
    epoch_summary_re = re.compile(
        r"Epoch\s+(\d+)/(\d+)\s*\|\s*train_loss=([0-9.]+)\s*\|\s*val_f1=([0-9.]+)\s*\|\s*merge=([0-9.]+)\s*\|\s*split=([0-9.]+)\s*\|\s*count_err=([0-9.]+)"
    )

    def redraw_curves(final=False):
        nonlocal plot_handle
        if not live_plots or not hist_train_loss:
            return
        if (not final) and (len(hist_train_loss) % max(1, int(live_plot_every)) != 0):
            return

        fig, ax = plt.subplots(1, 2, figsize=(12, 4))
        ax[0].plot(hist_train_loss, label='train_loss')
        ax[0].set_title('Train Loss')
        ax[0].legend()

        ax[1].plot(hist_val_f1, label='val_f1')
        ax[1].set_title('Val F1')
        ax[1].legend()

        if plot_handle is None:
            plot_handle = display(fig, display_id=True)
        else:
            plot_handle.update(fig)
        plt.close(fig)

    def ensure_epoch(epoch_num):
        nonlocal current_epoch, train_pbar, val_pbar
        if current_epoch == epoch_num:
            return

        if train_pbar is not None:
            train_pbar.close()
            train_pbar = None
        if val_pbar is not None:
            val_pbar.close()
            val_pbar = None

        current_epoch = epoch_num

    try:
        proc = subprocess.Popen(
            cmd,
            cwd=str(Path(cwd).resolve()),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
            env=env,
        )

        for raw in proc.stdout:
            if log_f is not None:
                log_f.write(raw)

            line = raw.rstrip('\n')

            mt = train_re.search(line)
            if mt:
                ep = int(mt.group(1)); ep_tot = int(mt.group(2))
                cur = int(mt.group(3)); tot = int(mt.group(4)); loss = float(mt.group(5))
                ensure_epoch(ep)
                if train_pbar is None or train_pbar.total != tot:
                    if train_pbar is not None:
                        train_pbar.close()
                    train_pbar = tqdm(total=tot, desc=f'Epoch {ep}/{ep_tot} [train]')
                if cur >= train_pbar.n:
                    train_pbar.update(cur - train_pbar.n)
                train_pbar.set_postfix(loss=f'{loss:.4f}')
                continue

            mv = val_re.search(line)
            if mv:
                ep = int(mv.group(1)); ep_tot = int(mv.group(2))
                cur = int(mv.group(3)); tot = int(mv.group(4))
                ensure_epoch(ep)
                if val_pbar is None or val_pbar.total != tot:
                    if val_pbar is not None:
                        val_pbar.close()
                    val_pbar = tqdm(total=tot, desc=f'Epoch {ep}/{ep_tot} [val]')
                if cur >= val_pbar.n:
                    val_pbar.update(cur - val_pbar.n)
                continue

            me = epoch_summary_re.search(line)
            if me:
                tl = float(me.group(3)); vf1 = float(me.group(4))
                hist_train_loss.append(tl)
                hist_val_f1.append(vf1)
                redraw_curves(final=False)
                print(line)
                continue

            if line.startswith('Epoch time:'):
                print(line)
                continue

            if 'Saved best:' in line or 'Loaded init checkpoint:' in line:
                print(line)
            elif not quiet_tqdm_lines and line:
                print(line)

        proc.wait()
    finally:
        redraw_curves(final=True)
        if train_pbar is not None:
            train_pbar.close()
        if val_pbar is not None:
            val_pbar.close()
        if log_f is not None:
            log_f.close()

    if proc.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {proc.returncode}: {cmd}')




In [ ]:
# Step 1: generate/re-generate two-stage ID splits
run_cmd([
    str(VENV_PYTHON),
    'tools/make_two_stage_ids.py',
    '--images_dir', 'trainable_pool/images',
    '--instances_dir', 'trainable_pool/instances',
    '--out_dir', str(SPLITS_DIR),
    '--cups_prefix', 'data_cups',
    '--val_split_stage1', '0.15',
    '--val_split_stage2', '0.15',
    '--seed', '42',
])


In [ ]:
# Show split sizes
for p in [
    SPLITS_DIR / 'stage1_pretrain_train_ids.txt',
    SPLITS_DIR / 'stage1_pretrain_val_ids.txt',
    SPLITS_DIR / 'stage2_finetune_train_ids.txt',
    SPLITS_DIR / 'stage2_finetune_val_ids.txt',
]:
    n = len([x for x in p.read_text(encoding='utf-8').splitlines() if x.strip()])
    print(f'{p}: {n}')


In [ ]:
# Step 2: Stage 1 training (Mask2Former pretrain on non-cups)
run_cmd([str(VENV_PYTHON), '-m', 'pip', 'install', '-U', 'transformers', 'accelerate', 'datasets', 'scikit-image'])

import cv2, numpy as np, torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoImageProcessor, Mask2FormerForUniversalSegmentation
from colonyseg.metrics.instance_metrics import instance_scores

BASE='facebook/mask2former-swin-small-coco-instance'
IMG_SIZE=320; BS=1; E1=8; E2=8; MAX_INST=256
device='cuda' if torch.cuda.is_available() else 'cpu'
processor=AutoImageProcessor.from_pretrained(BASE)

def _ids(p): return [x.strip() for x in Path(p).read_text(encoding='utf-8').splitlines() if x.strip()]
def _find(image_id):
    for ext in ('.png','.jpg','.jpeg','.tif','.tiff','.bmp'):
        p=Path('trainable_pool/images')/f'{image_id}{ext}'
        if p.exists(): return p
    raise FileNotFoundError(image_id)
def _dense(m):
    u=np.unique(m); u=u[u!=0]; out=np.zeros_like(m,dtype=np.int32)
    for i,v in enumerate(u,1): out[m==v]=i
    return out

class DS(Dataset):
    def __init__(self, ids): self.ids=ids
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        image_id=self.ids[i]
        img=cv2.imread(str(_find(image_id)), cv2.IMREAD_COLOR)
        msk=cv2.imread(str(Path('trainable_pool/instances')/f'{image_id}.png'), cv2.IMREAD_UNCHANGED)
        img=cv2.cvtColor(img, cv2.COLOR_BGR2RGB); img=cv2.resize(img,(IMG_SIZE,IMG_SIZE), interpolation=cv2.INTER_AREA)
        msk=cv2.resize(msk.astype(np.int32),(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST); msk=_dense(msk)
        uniq=[int(x) for x in np.unique(msk) if x>0]
        if len(uniq)>MAX_INST:
            keep=set(uniq[:MAX_INST]); msk=np.where(np.isin(msk,list(keep)),msk,0).astype(np.int32); msk=_dense(msk); uniq=[int(x) for x in np.unique(msk) if x>0]
        mp={0:0};
        for u in uniq: mp[u]=1
        enc=processor(images=img, segmentation_maps=msk, instance_id_to_semantic_id=mp, return_tensors='pt', size={'height':IMG_SIZE,'width':IMG_SIZE}, ignore_index=0)
        return {'pv':enc['pixel_values'][0], 'pm':enc['pixel_mask'][0], 'ml':enc['mask_labels'][0], 'cl':enc['class_labels'][0], 'gt':torch.from_numpy(msk.astype(np.int32))}

def _coll(b):
    return {'pv':torch.stack([x['pv'] for x in b],0), 'pm':torch.stack([x['pm'] for x in b],0), 'ml':[x['ml'] for x in b], 'cl':[x['cl'] for x in b], 'gt':[x['gt'] for x in b]}

def _model():
    return Mask2FormerForUniversalSegmentation.from_pretrained(BASE, num_labels=2, id2label={0:'background',1:'colony'}, label2id={'background':0,'colony':1}, ignore_mismatched_sizes=True).to(device)

def _decode(out,h,w,thr=0.5):
    r=processor.post_process_instance_segmentation(out,target_sizes=[(h,w)],threshold=thr,mask_threshold=0.5,overlap_mask_area_threshold=0.8)[0]['segmentation']
    if isinstance(r,torch.Tensor): r=r.detach().cpu().numpy()
    r=r.astype(np.int32); r[r<0]=0
    return _dense(r)

def _train(train_ids,val_ids,epochs,lr,init=None,save=None,log=None):
    model=_model();
    if init and Path(init).exists(): model.load_state_dict(torch.load(init,map_location='cpu')['model'])
    tr=DataLoader(DS(train_ids),batch_size=BS,shuffle=True,num_workers=0,collate_fn=_coll)
    va=DataLoader(DS(val_ids),batch_size=1,shuffle=False,num_workers=0,collate_fn=_coll)
    opt=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=1e-4)
    best=-1; lf=Path(log).open('w',encoding='utf-8')
    for e in range(1,epochs+1):
        model.train(); ls=0.; n=0
        for b in tr:
            out=model(pixel_values=b['pv'].to(device), pixel_mask=b['pm'].to(device), mask_labels=[x.to(device) for x in b['ml']], class_labels=[x.to(device) for x in b['cl']])
            loss=out.loss; opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); ls+=float(loss.item()); n+=1
        tl=ls/max(1,n)
        model.eval(); mets=[]
        with torch.no_grad():
            for b in va:
                out=model(pixel_values=b['pv'].to(device), pixel_mask=b['pm'].to(device))
                gt=b['gt'][0].numpy().astype(np.int32); pr=_decode(out,gt.shape[0],gt.shape[1],0.5); mets.append(instance_scores(gt,pr,0.5))
        f1=float(np.mean([m['f1'] for m in mets])) if mets else 0.; me=float(np.mean([m['merge'] for m in mets])) if mets else 0.; sp=float(np.mean([m['split'] for m in mets])) if mets else 0.; ce=float(np.mean([m['count_err'] for m in mets])) if mets else 0.
        if f1>best: best=f1; torch.save({'model':model.state_dict(),'best_f1':best}, save)
        line=f'Epoch {e}/{epochs} | train_loss={tl:.4f} | val_f1={f1:.4f} | merge={me:.3f} | split={sp:.3f} | count_err={ce:.3f}'
        print(line); lf.write(line+'\n')
    lf.close(); return model

s1_tr=_ids(SPLITS_DIR/'stage1_pretrain_train_ids.txt'); s1_va=_ids(SPLITS_DIR/'stage1_pretrain_val_ids.txt')
_train(s1_tr,s1_va,E1,3e-5,None,STAGE1_BEST,LOG_STAGE1)


In [ ]:
# Verify Stage 1 checkpoint exists
print('stage1 exists:', STAGE1_BEST.exists(), STAGE1_BEST)
if not STAGE1_BEST.exists():
    raise FileNotFoundError(STAGE1_BEST)


In [ ]:
# Step 3: Stage 2 training (finetune on cups only)
s2_tr=_ids(SPLITS_DIR/'stage2_finetune_train_ids.txt'); s2_va=_ids(SPLITS_DIR/'stage2_finetune_val_ids.txt')
_train(s2_tr,s2_va,E2,1e-5,STAGE1_BEST,STAGE2_BEST,LOG_STAGE2)


In [ ]:
# Verify Stage 2 checkpoints
print('stage2 exists:', STAGE2_BEST.exists(), STAGE2_BEST)
if not STAGE2_BEST.exists():
    raise FileNotFoundError(STAGE2_BEST)


In [ ]:
# Training curves from logs
import re
import numpy as np
import matplotlib.pyplot as plt


def parse_train_log(path):
    path = Path(path)
    if not path.exists():
        print(f'log missing: {path}')
        return None

    epoch, train_loss = [], []
    val_f1, val_merge, val_split, val_count = [], [], [], []

    pat = re.compile(
        r"Epoch\s+(\d+)/(\d+)\s*\|\s*train_loss=([0-9.]+)\s*\|\s*val_f1=([0-9.]+)\s*\|\s*merge=([0-9.]+)\s*\|\s*split=([0-9.]+)\s*\|\s*count_err=([0-9.]+)"
    )

    for ln in path.read_text(encoding='utf-8', errors='ignore').splitlines():
        m = pat.search(ln)
        if not m:
            continue
        epoch.append(int(m.group(1)))
        train_loss.append(float(m.group(3)))
        val_f1.append(float(m.group(4)))
        val_merge.append(float(m.group(5)))
        val_split.append(float(m.group(6)))
        val_count.append(float(m.group(7)))

    if not epoch:
        print(f'no parsed epoch lines in: {path}')
        return None

    return {
        'epoch': np.array(epoch),
        'train_loss': np.array(train_loss),
        'val_f1': np.array(val_f1),
        'val_merge': np.array(val_merge),
        'val_split': np.array(val_split),
        'val_count_err': np.array(val_count),
        'path': path,
    }


def plot_stage_curves(data, title):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))

    ax[0].plot(data['train_loss'], label='train_loss')
    ax[0].set_title('Train Loss')
    ax[0].legend()

    ax[1].plot(data['val_f1'], label='val_f1')
    ax[1].set_title('Val F1')
    ax[1].legend()

    print(title)
    plt.show()


stage1 = parse_train_log(LOG_STAGE1)
stage2 = parse_train_log(LOG_STAGE2)

if stage1 is not None:
    print('Stage1 epochs:', len(stage1['epoch']), 'best val_f1:', float(stage1['val_f1'].max()))
    plot_stage_curves(stage1, 'Stage 1: Pretrain (non-cups)')

if stage2 is not None:
    print('Stage2 epochs:', len(stage2['epoch']), 'best val_f1:', float(stage2['val_f1'].max()))
    plot_stage_curves(stage2, 'Stage 2: Finetune (cups)')



## Notes

- Mask2Former is memory heavy; reduce `IMG_SIZE`/`BS` if OOM.
- `MAX_INST` limits per-image instances to stabilize training.


In [ ]:
# Final visualization cell
import cv2, torch, numpy as np, matplotlib.pyplot as plt
from transformers import AutoImageProcessor, Mask2FormerForUniversalSegmentation

processor=AutoImageProcessor.from_pretrained(BASE)
model=_model(); model.load_state_dict(torch.load(STAGE2_BEST,map_location='cpu')['model']); model.eval()
img=cv2.imread('IMG_4377.jpg', cv2.IMREAD_COLOR); img_rgb=cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
enc=processor(images=img_rgb, return_tensors='pt', size={'height':IMG_SIZE,'width':IMG_SIZE})
with torch.no_grad(): out=model(pixel_values=enc['pixel_values'].to(device), pixel_mask=enc['pixel_mask'].to(device))
pr=_decode(out, IMG_SIZE, IMG_SIZE, 0.5)
pr=cv2.resize(pr.astype(np.int32),(img_rgb.shape[1],img_rgb.shape[0]),interpolation=cv2.INTER_NEAREST)
edges=cv2.Canny((pr>0).astype(np.uint8)*255,50,150)>0
vis=img_rgb.copy(); vis[edges]=[255,0,0]
plt.figure(figsize=(14,6))
plt.subplot(1,2,1); plt.title('Input'); plt.imshow(img_rgb); plt.axis('off')
plt.subplot(1,2,2); plt.title(f'Mask2Former instances: {int(np.max(pr))}'); plt.imshow(vis); plt.axis('off')
plt.tight_layout(); plt.show()
